In [31]:
import pandas as pd
import os

folder_path = "data/ISPU"

files = {
    "data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data-2024.csv": 2024,
    "data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data-2025.csv": 2025
}

for file, year in files.items():
    df = pd.read_csv(os.path.join(folder_path, file))

    df["year"] = year

    df.rename(columns={
        "bulan": "month",
        "tanggal": "day"
    }, inplace=True)

    df["tanggal"] = pd.to_datetime(
        df[["year", "month", "day"]],
        errors="coerce"
    )

    df.drop(columns=["year", "month", "day"], inplace=True)

    df = df[["tanggal"] + [c for c in df.columns if c != "tanggal"]]

    df.to_csv(os.path.join(folder_path, file), index=False)


In [32]:
df.head()
df["tanggal"].min(), df["tanggal"].max()
df.dtypes

tanggal                      datetime64[ns]
periode_data                          int64
stasiun                              object
pm_sepuluh                          float64
pm_duakomalima                      float64
sulfur_dioksida                     float64
karbon_monoksida                    float64
ozon                                float64
nitrogen_dioksida                   float64
max                                 float64
parameter_pencemar_kritis            object
kategori                             object
dtype: object

In [33]:
COLUMN_MAPPING = {
    "lokasi_spku": "stasiun",

    "pm_sepuluh": "pm10",
    "pm_10": "pm10",

    "pm_duakomalima": "pm2.5",
    "pm25": "pm2.5",

    "so2": "so2",
    "no2": "no2",
    "co": "co",
    "o3": "o3",
    "sulfur_dioksida": "so2",
    "karbon_monoksida": "co",
    "ozon": "o3",
    "nitrogen_dioksida": "no2",

    "categori": "kategori",
    
    "parameter_pencemar_kritis": "critical"
}


In [34]:
import pandas as pd
import os

folder_path = "data/ISPU"

files = os.listdir(folder_path)

for file in files:
    if not file.endswith(".csv"):
        continue

    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)

    # lowercase + strip spasi
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
    )

    # rename pakai mapping (yang ada saja)
    rename_dict = {
        col: COLUMN_MAPPING[col]
        for col in df.columns
        if col in COLUMN_MAPPING
    }

    df.rename(columns=rename_dict, inplace=True)

    df.to_csv(file_path, index=False)

print("Semua nama kolom sudah sama")


Semua nama kolom sudah sama


In [35]:
REQUIRED_COLUMNS = ["pm10", "pm2.5", "so2", "no2", "co", "o3"]

for file in files:
    if not file.endswith(".csv"):
        continue

    df = pd.read_csv(os.path.join(folder_path, file))

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    df.to_csv(os.path.join(folder_path, file), index=False)


In [36]:
import pandas as pd
import os

folder_path = "data/ISPU"

def clean_tanggal(x):
    if pd.isna(x):
        return pd.NaT

    try:
        x_num = float(x)
        if x_num > 30000: 
            return pd.to_datetime(x_num, unit="D", origin="1899-12-30")
    except:
        pass

    return pd.to_datetime(x, dayfirst=True, errors="coerce")

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)

    if "tanggal" not in df.columns:
        continue

    df["tanggal"] = df["tanggal"].apply(clean_tanggal)
    df["tanggal"] = df["tanggal"].dt.strftime("%Y-%m-%d")

    df.to_csv(file_path, index=False)


print("format tanggal sudah rapi")


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_27076\2688791125.py:17: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_27076\2688791125.py:17: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_27076\2688791125.py:17: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_27076\2688791125.py:17: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified

format tanggal sudah rapi


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_27076\2688791125.py:17: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")


In [37]:
import pandas as pd
import os

folder_path = "data/ISPU"

dfs = []

files = [
    f for f in os.listdir(folder_path)
    if f.endswith(".csv")
]

for file in files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

df_all["tanggal"] = pd.to_datetime(df_all["tanggal"], errors="coerce")

df_all = (
    df_all
    .sort_values(["tanggal", "stasiun"])
    .reset_index(drop=True)
)

output_path = "data/ISPU/ispu_dki_2010_2025.csv"
df_all.to_csv(output_path, index=False)

print("Data telah digabungkan")


Data telah digabungkan
